In [2]:
from env.EnvMultipleStock_trade import StockEnvTrade
from env.EnvMultipleStock_train import StockEnvTrain
from stable_baselines.common.vec_env import DummyVecEnv

from stable_baselines import A2C, PPO2, DDPG

def train_model(model_name, env_train, timesteps=50000):

    if model_name == "A2C":
        model = A2C("MlpPolicy", env_train, verbose=0)

    elif model_name == "PPO":
        model = PPO2("MlpPolicy", env_train, verbose=0)

    elif model_name == "DDPG":
        model = DDPG("MlpPolicy", env_train, verbose=0)

    else:
        raise ValueError("Invalid model")

    model.learn(total_timesteps=timesteps)
    return model
def trade_model(model, env_trade):
    obs = env_trade.reset()
    account_values = []

    for _ in range(len(env_trade.envs[0].df.index.unique())):
        action, _ = model.predict(obs)
        obs, reward, done, info = env_trade.step(action)

        account_values.append(env_trade.envs[0].asset_memory[-1])

    df_trade = pd.DataFrame(account_values, columns=["account_value"])

    return df_trade
def run_single_model_strategy(
    df,
    unique_trade_date,
    model_name="A2C",
    rebalance_window=63,
    validation_window=63
):

    print(f"Starting {model_name} Strategy...")

    df = df.sort_values(["datadate", "tic"])

    portfolio_value = []

    for i in range(rebalance_window + validation_window, len(unique_trade_date), rebalance_window):

        print(f"\n===== {model_name} Iteration {i} =====")

        # -------------------------
        # Define time windows (SAME as ensemble)
        # -------------------------
        train_start = 0
        train_end = i - validation_window
        trade_start = i
        trade_end = i + rebalance_window

        train_dates = unique_trade_date[train_start:train_end]
        trade_dates = unique_trade_date[trade_start:trade_end]

        df_train = df[df.datadate.isin(train_dates)].copy()
        df_trade = df[df.datadate.isin(trade_dates)].copy()

        # 🚨 SAFETY: skip empty windows
        if len(df_train) == 0 or len(df_trade) == 0:
            print("⚠️ Skipping empty window")
            continue

        # -------------------------
        # FIX INDEX (CRITICAL)
        # -------------------------
        df_train = df_train.sort_values(["datadate", "tic"])
        df_trade = df_trade.sort_values(["datadate", "tic"])

        df_train.index = df_train.datadate.factorize()[0]
        df_trade.index = df_trade.datadate.factorize()[0]

        # -------------------------
        # Sanity check (VERY IMPORTANT)
        # -------------------------
        try:
            print("Train first day shape:", df_train.loc[0].shape)
        except:
            print("❌ Index issue in train")
            continue

        # -------------------------
        # Create environments
        # -------------------------
        env_train = DummyVecEnv([lambda: StockEnvTrain(df_train)])   # ✅ FIX
        env_trade = DummyVecEnv([lambda: StockEnvTrade(df_trade)])   # ✅ OK

        # -------------------------
        # Train ONLY ONE MODEL
        # -------------------------
        model = train_model(model_name, env_train)

        # -------------------------
        # Trade
        # -------------------------
        df_trade_result = trade_model(model, env_trade)

        portfolio_value.append(df_trade_result)

    # -------------------------
    # Combine results
    # -------------------------
    df_result = pd.concat(portfolio_value, ignore_index=True)

    return df_result

In [4]:
from stable_baselines import A2C, PPO2, DDPG
import pandas as pd
import numpy as np

df = pd.read_csv("new_done_data.csv")

df = df.sort_values(["datadate", "tic"])
df.index = df.datadate.factorize()[0]

unique_trade_date = df.datadate.unique()

a2c_full = run_single_model_strategy(
    df=df,
    unique_trade_date = unique_trade_date,
    model_name="A2C"
)

ppo_full = run_single_model_strategy(
    df=df,
    unique_trade_date = unique_trade_date,
    model_name="PPO"
)

ddpg_full = run_single_model_strategy(
    df=df,
    unique_trade_date = unique_trade_date,
    model_name="DDPG"
)
#a2c_full.to_csv("a2c_full.csv", index=False)
#ppo_full.to_csv("ppo_full.csv", index=False)
#ddpg_full.to_csv("ddpg_full.csv", index=False)

Starting A2C Strategy...

===== A2C Iteration 126 =====
Train first day shape: (30, 13)
previous_total_asset:1000000
end_total_asset:991965.1428751805
total_reward:-8034.857124819537
total_cost:  6649.253423572048
total trades:  1625
Sharpe:  -0.04185192674747745

===== A2C Iteration 189 =====
Train first day shape: (30, 13)
previous_total_asset:1000000
end_total_asset:1071410.2708036439
total_reward:71410.27080364386
total_cost:  4357.014192340559
total trades:  1175
Sharpe:  0.43311887522981946

===== A2C Iteration 252 =====
Train first day shape: (30, 13)
previous_total_asset:1000000
end_total_asset:1012177.7427317282
total_reward:12177.742731728242
total_cost:  3191.210794490554
total trades:  830
Sharpe:  0.08296863978438927

===== A2C Iteration 315 =====
Train first day shape: (30, 13)
previous_total_asset:1000000
end_total_asset:1025200.8588137873
total_reward:25200.858813787345
total_cost:  4909.331610996033
total trades:  1253
Sharpe:  0.18185232070579857

===== A2C Iteration 

previous_total_asset:1000000
end_total_asset:979096.2775579555
total_reward:-20903.722442044527
total_cost:  7969.748285904813
total trades:  1070
Sharpe:  -0.09719291421477066

===== A2C Iteration 2331 =====
Train first day shape: (30, 13)
previous_total_asset:1000000
end_total_asset:886880.2752303585
total_reward:-113119.72476964153
total_cost:  5467.279229692493
total trades:  971
Sharpe:  -0.3301158161516101

===== A2C Iteration 2394 =====
Train first day shape: (30, 13)
previous_total_asset:1000000
end_total_asset:962710.238042095
total_reward:-37289.76195790502
total_cost:  7530.012301096734
total trades:  1447
Sharpe:  -0.07783680495929876

===== A2C Iteration 2457 =====
Train first day shape: (30, 13)
previous_total_asset:1000000
end_total_asset:1094523.3884522847
total_reward:94523.38845228474
total_cost:  8081.098436686908
total trades:  1402
Sharpe:  0.2987307595303609

===== A2C Iteration 2520 =====
Train first day shape: (30, 13)
previous_total_asset:1000000
end_total_asse

previous_total_asset:1000000
end_total_asset:872918.6193295075
total_reward:-127081.38067049254
total_cost:  6599.918779199698
total trades:  1171
Sharpe:  -0.38778974192431437

===== PPO Iteration 1512 =====
Train first day shape: (30, 13)
previous_total_asset:1000000
end_total_asset:1063251.9968725634
total_reward:63251.99687256338
total_cost:  8358.828949877454
total trades:  1469
Sharpe:  0.4202337349714942

===== PPO Iteration 1575 =====
Train first day shape: (30, 13)
previous_total_asset:1000000
end_total_asset:1050373.0372697045
total_reward:50373.03726970451
total_cost:  5739.53355911075
total trades:  1171
Sharpe:  0.29731853741525754

===== PPO Iteration 1638 =====
Train first day shape: (30, 13)
previous_total_asset:1000000
end_total_asset:989166.060071694
total_reward:-10833.939928306034
total_cost:  7503.307798557493
total trades:  1557
Sharpe:  -0.029612280563204123

===== PPO Iteration 1701 =====
Train first day shape: (30, 13)
previous_total_asset:1000000
end_total_ass

previous_total_asset:1000000
end_total_asset:1071661.4706412263
total_reward:71661.47064122627
total_cost:  2881.9460090088232
total trades:  945
Sharpe:  0.2649391884504301

===== DDPG Iteration 504 =====
Train first day shape: (30, 13)
previous_total_asset:1000000
end_total_asset:1001860.1185448485
total_reward:1860.1185448485194
total_cost:  4191.73093143642
total trades:  915
Sharpe:  0.015406605935953764

===== DDPG Iteration 567 =====
Train first day shape: (30, 13)
previous_total_asset:1000000
end_total_asset:982153.6565824148
total_reward:-17846.343417585245
total_cost:  2604.958100112012
total trades:  943
Sharpe:  -0.0650242251625855

===== DDPG Iteration 630 =====
Train first day shape: (30, 13)
previous_total_asset:1000000
end_total_asset:895209.6930118698
total_reward:-104790.3069881302
total_cost:  3961.8905706763608
total trades:  1110
Sharpe:  -0.29394693128346944

===== DDPG Iteration 693 =====
Train first day shape: (30, 13)
previous_total_asset:1000000
end_total_asse

previous_total_asset:1000000
end_total_asset:912524.8814081221
total_reward:-87475.11859187786
total_cost:  2067.228286898022
total trades:  968
Sharpe:  -0.4443144890252783

===== DDPG Iteration 2709 =====
Train first day shape: (30, 13)
previous_total_asset:1000000
end_total_asset:1127643.4060147768
total_reward:127643.40601477679
total_cost:  2006.5676637591794
total trades:  1034
Sharpe:  0.5156190592994765

===== DDPG Iteration 2772 =====
Train first day shape: (30, 13)
previous_total_asset:1000000
end_total_asset:974662.5521406682
total_reward:-25337.44785933185
total_cost:  2925.4105915354044
total trades:  661
Sharpe:  -0.13352945068237498

===== DDPG Iteration 2835 =====
Train first day shape: (30, 13)
previous_total_asset:1000000
end_total_asset:1050818.6524273474
total_reward:50818.65242734738
total_cost:  2779.496984114342
total trades:  827
Sharpe:  0.24325227128071417

===== DDPG Iteration 2898 =====
Train first day shape: (30, 13)
previous_total_asset:1000000
end_total_a